# RE-VISION · Colab GPU 학습 + Google Drive

이 노트북은 **Drive 데이터 → Colab 로컬 복사 → 그룹 분할 → GPU 학습 → epoch 저장/재개 → ONNX 검증 → Drive 결과 저장**을 실행합니다.
기본 모델은 ImageNet 사전학습 MobileNetV3-Small을 정상/불량 이진 분류로 fine-tuning하는 기준선입니다. 최종 Scratch/Dent 위치 검출·분할 모델은 별도로 개발합니다.

1. Colab 메뉴 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다.
2. Drive `내 드라이브/RE-VISION/datasets/v001/`에 `manifest.jsonl`과 이미지 폴더를 준비합니다.
3. 아래 `PROJECT_ROOT`, `DATASET_SOURCE`, `RUN_ID`를 확인하고 순서대로 실행합니다.
4. 중단 후 새 런타임에서는 동일 데이터·RUN_ID를 사용하고 `RESUME=True`로 바꿉니다.

노트북 사본을 본인 Drive에 저장해 설정을 보관하세요. 토큰이나 서비스 계정 키는 필요 없으며 Drive 마운트 때 본인 계정으로 인증합니다.
[데이터 형식과 상세 안내](https://github.com/KiHyeonLee1121/RE-VISION/blob/main/docs/colab-drive.md)

## 1. 코드와 선택 의존성 설치
`REPO_REF`는 최초에는 `main`, 중단한 학습 재개 시에는 Drive의 `runs/<RUN_ID>/run.json`에 기록된 `code_revision`으로 설정하세요. setup 셀은 한 런타임에서 한 번만 실행합니다.

In [ ]:
import importlib
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/KiHyeonLee1121/RE-VISION.git"
REPO_REF = "main"  # 재개 시 기존 run.json의 code_revision 사용

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 이상의 Colab 런타임이 필요합니다.")
if any(name == "revision" or name.startswith("revision.") for name in sys.modules):
    raise RuntimeError(
        "setup은 이미 실행되었습니다. 다음 셀부터 진행하거나 런타임을 다시 시작하세요."
    )
REPO_DIR = Path(tempfile.mkdtemp(prefix="revision-code-", dir="/content")) / "repo"
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR, check=True)
CODE_REVISION = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_DIR) + "[colab]"], check=True
)
sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
print("사용 코드:", CODE_REVISION)

## 2. GPU 확인
Colab Pro에서도 할당 GPU는 달라질 수 있습니다. CUDA가 없으면 학습 전에 멈춥니다. GPU 메모리가 부족하면 새 RUN_ID로 `batch_size`를 낮춰 시작하세요.

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. 런타임 유형을 GPU로 변경한 뒤 1번부터 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 3. Drive 연결 및 경로 설정
`DATASET_SOURCE`는 폴더 또는 ZIP입니다. ZIP 최상위에도 `manifest.jsonl`이 바로 있어야 합니다. 기존 Drive 원본은 수정하지 않습니다.
`RUN_ID`가 같은 디렉토리에서 두 런타임을 동시에 실행하지 마세요.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/RE-VISION")
DATASET_SOURCE = PROJECT_ROOT / "datasets" / "v001"  # 또는 PROJECT_ROOT / "datasets/v001.zip"
RUN_ID = "mobilenet-v001-run01"
RESUME = False  # 중단한 학습: 같은 RUN_ID + True

if not RUN_ID or any(
    c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in RUN_ID
):
    raise ValueError("RUN_ID에는 영문, 숫자, 밑줄, 하이픈만 사용하세요.")
if not DATASET_SOURCE.exists():
    raise FileNotFoundError(f"데이터셋 경로를 확인하세요: {DATASET_SOURCE}")
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
WORK_DIR = Path("/content/revision-work") / RUN_ID
print("학습 결과 저장:", RUN_DIR)

## 4. 학습 설정
MobileNetV3-Small 전체를 fine-tuning합니다. `epochs`는 추가 횟수가 아니라 **총 목표 epoch 수**입니다. 학습 재개 시 epochs를 늘릴 수 있고, 다른 학습 설정과 코드·데이터가 달라지면 재개를 거부합니다.

In [ ]:
from dataclasses import asdict, replace

from revision.ml.train_torch import TrainingConfig

CONFIG = TrainingConfig.from_toml(REPO_DIR / "configs/train.colab.toml")
CONFIG = replace(CONFIG, epochs=20, batch_size=16)
print(asdict(CONFIG))

## 5. Drive → 작업 디스크 복사·검증·분할
이미지를 매 batch마다 Drive에서 읽지 않고 로컬 디스크로 복사합니다. 중복 이미지와 파일 누락을 확인하고, 같은 부품/촬영 세션은 하나의 split에만 배치합니다. train/val에 정상·불량이 모두 포함되지 않으면 데이터 수집 구성을 확인하세요.

In [ ]:
from revision.data.staging import prepare_splits, stage_dataset

LOCAL_MANIFEST = stage_dataset(DATASET_SOURCE, Path("/content/revision-cache"))
SPLIT_DIR = LOCAL_MANIFEST.parent.parent / "splits"
SPLIT_SUMMARY = prepare_splits(LOCAL_MANIFEST, SPLIT_DIR, seed=CONFIG.seed)
for name, summary in SPLIT_SUMMARY["splits"].items():
    print(name, {k: v for k, v in summary.items() if k != "sample_ids"})
print("데이터 버전:", SPLIT_SUMMARY["dataset_signature"])

## 6. 학습 / 이어서 학습
매 epoch 완료 후 모델·optimizer·AMP scaler·난수 상태·최고 모델·학습 이력을 Drive에 저장합니다. 중간에 끊기면 마지막 **저장이 완료된 epoch 다음부터** 재개하며, 진행 중이던 epoch은 다시 수행합니다.
처음 학습 시 사전학습 가중치를 내려받습니다. 체크포인트 재개는 가중치를 다시 다운로드하지 않습니다.

In [ ]:
from revision.ml.artifacts import atomic_json
from revision.ml.train_torch import train

TRAINING_RESULT = train(
    SPLIT_DIR / "train.jsonl",
    SPLIT_DIR / "val.jsonl",
    CONFIG,
    RUN_DIR,
    WORK_DIR,
    resume=RESUME,
    code_revision=CODE_REVISION,
)
atomic_json(RUN_DIR / "split-summary.json", SPLIT_SUMMARY)
print(TRAINING_RESULT)

## 7. 최고 validation 모델 → ONNX → 기존 추론 코드 검증 → Drive 저장
validation loss가 가장 낮았던 모델을 내보냅니다. 실제 validation 이미지 최대 5장에서 PyTorch와 ONNX Runtime의 점수를 비교하고, 통과한 모델 묶음을 Drive에 저장합니다. 테스트 세트는 여기서 모델 선택에 사용하지 않습니다.

In [ ]:
from revision.ml.export_torch import export_best

EXPORT_DIR = export_best(RUN_DIR, SPLIT_DIR / "val.jsonl", WORK_DIR)
print("모델 폴더:", EXPORT_DIR)
print("다운로드할 파일:", EXPORT_DIR / "model-bundle.zip")
print("run.json과 latest.json도 Drive에서 저장 상태를 확인하세요.")

## 8. Edge / 노트북 실행 코드에 연결
Drive의 `model-bundle.zip`을 실행할 컴퓨터로 내려받아 저장소 `models/weights/<모델이름>/`에 풉니다. `model.onnx`, `manifest.json`, `metrics.json`을 함께 보관하세요.

`configs/local.toml` 또는 replay 설정에 아래 경로를 넣습니다.
```toml
model_manifest = "../models/weights/<모델이름>/manifest.json"
```
기존 `OnnxBinaryClassifier`가 동일 전처리와 model checksum을 검증하여 불러옵니다. 실제 장비의 PASS/FAIL threshold는 별도 검증이 필요합니다.

Drive 마운트에 기록된 파일이 서버에 완전히 동기화되는 시간은 별개입니다. 종료 전 Drive 웹 화면에서 체크포인트와 모델 묶음을 확인하세요.
GPU 학습 결과나 실데이터 성능은 이 저장소에 미리 포함되어 있지 않습니다.